[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Why SQLAlchemy


## What you will be able to do

Say what SQLAlchemy is for, and when it earns its install. Build a query in Python that changes with
the options a page was given, without pasting a single value into SQL text. Read the SQL SQLAlchemy
sends, and compile the same query for PostgreSQL or SQL Server with no server running. Run SQL you
wrote yourself with its values bound, read rows back as objects of a class of your own, and
recognize the first errors people meet when they start.


## The idea

### The problem

A college keeps its students in a database: a name, an email, a program such as Biology or History,
and the day each student started. The registrar's search page has three boxes, all of them optional:
a program, a first day, and part of a name. The code behind the page writes SQL as text. It starts
from `SELECT name, program, started_on FROM students`, and then three `if` statements each add a
condition for a box that was filled in, `WHERE` before the first and `AND` before the rest, with
the value from the box pasted into the text.

It works for `Biology` and for `Okafor`. Then someone searches for `O'Brien`, and the apostrophe ends
the quoted text early, so the database is handed SQL that makes no sense and refuses it. A search
typed on purpose could do worse, and be run as SQL of its own. Placeholders keep the values out of
the text, but the three `if` statements stay: a list of conditions and a list of values that have to
be kept in step by hand, a `WHERE` that must appear once and only once, and a query nobody can look
at without running it. The rows come back as tuples, so the program says `row[1]` and hopes that is
still the program, and every date arrives as text. And the day the college moves to PostgreSQL,
every `?` in that text has to be found and rewritten.

### What SQLAlchemy is

> **SQLAlchemy** is a Python library for working with relational databases, in two layers. **Core**
> writes SQL as Python objects: a **`Table`** describes a table and its columns, **`select()`**,
> **`insert()`**, **`update()`** and **`delete()`** build statements, and every value in a statement
> travels as a **bind parameter**, next to the SQL and never inside it. An **engine**, made by
> **`create_engine()`** from a **URL**, knows how to reach one database, and its **dialect** turns a
> statement into the SQL that database speaks. **`text()`** runs SQL you wrote yourself, with its
> values still bound. The **ORM**, built on Core, maps a Python class to a table, so rows come back
> as objects, and a **`Session`** keeps track of the objects a program reads and changes, and writes
> the changes back.

### Why it works that way

- **A statement is an object until it runs.** `select(students)` can be built, added to and printed
  before any database sees it. Every `.where()` returns a new statement with one more condition,
  joined to the others by `AND`, so the three `if` statements become three `.where()` calls, and
  the `WHERE` and the `AND`s are SQLAlchemy's problem.
- **Values never become SQL text.** Comparing a column with a value makes a bind parameter, and the
  database driver sends the value separately. An apostrophe in a name is just a character, and a
  search box cannot change the SQL it is fed into.
- **The dialect writes the SQL.** The same statement becomes `?` placeholders for SQLite,
  `%(name)s` for PostgreSQL's driver, and `TOP 5` instead of `LIMIT 5` for SQL Server, and code
  that builds statements does not change when the database does.
- **Types travel both ways.** A column declared as a `Date` turns a `datetime.date` into what the
  database stores, and turns what comes back into a `date` again.
- **The ORM stands on Core.** Every query the ORM runs is a Core statement underneath, which is why
  this guide learns Core first, and why the ORM's SQL can always be printed and read.
- **There is one way to run anything.** In SQLAlchemy 2.0 a statement runs through a connection or
  a session with `execute`. Text is SQL only when `text()` says so, and the older habits that
  tutorials still show, such as `engine.execute`, are gone.

### Where this shows up

SQLModel, the subject of the **SQLModel, Deep Dive** guide, is SQLAlchemy's ORM with Pydantic models
on top, and Flask-SQLAlchemy wraps SQLAlchemy for the Flask applications of the **Flask, Deep Dive**
guide. A web service of the kind the **FastAPI, Deep Dive** guide builds usually reaches a relational
database through SQLAlchemy, Pandas' `read_sql` accepts a SQLAlchemy engine and returns a DataFrame,
the subject of the **Pandas, Deep Dive** guide, and Apache Airflow, in the
**Apache Airflow, Deep Dive** guide, keeps its own records in a database through SQLAlchemy. The
**sqlite3, Deep Dive** guide works one level down, with the database engine this guide runs on and
the standard library's driver for it.

### The vocabulary of SQLAlchemy

This guide works from the bottom up: Core first, the ORM on top of it, and then the tools around
both. The tables below are the map, and their last column names the notebook that covers each term
in depth.

Core, SQL as Python objects:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| Engine | the object that knows how to reach one database, and keeps its connections | `create_engine("sqlite:///scratch/college.db")` | this notebook, and **Engines and URLs** |
| URL | a string naming the database, the driver and where to find them | `postgresql+psycopg://host/college` | **Engines and URLs** |
| Dialect and driver | the SQL a database speaks, and the library that talks to it | SQLite through `sqlite3` | this notebook, **Engines and URLs**, and **Four Databases, One Codebase** |
| Connection pool | the connections an engine keeps open, to hand out again | `QueuePool` | **Engines and URLs** |
| Connection and transaction | one conversation with the database, and a group of changes saved together | `with engine.begin() as conn:` | **Connections and Transactions** |
| Result and row | what a statement hands back, and one record in it | `result.all()`, `row.name` | **Reading Results** |
| `Table` and `MetaData` | a description of a table, and the collection of them | `Table("students", metadata, ...)` | this notebook, and **Tables and Metadata** |
| Statement | a `select`, `insert`, `update` or `delete`, built as an object | `select(students).where(...)` | this notebook, and **SQL Expressions** |
| Bind parameter | a value sent beside the SQL, never inside it | `:program_1` | this notebook, and **SQL Expressions** |
| `text()` | SQL written by hand, still with bound values | `text("SELECT ... WHERE id = :id")` | this notebook |

The ORM, rows as objects:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| Mapped class | a Python class whose objects are the rows of a table | `class Student(Base)` | this notebook, and **Declarative Models** |
| `Mapped` and `mapped_column` | how a class declares its columns and their types | `name: Mapped[str]` | **Declarative Models** |
| Column type | how a Python value is stored, and read back | `Date`, `Numeric(10, 2)` | **Column Types** |
| `Session` | the object that loads rows as objects and saves the changes made to them | `with Session(engine) as session:` | this notebook, and **The Session** |
| Flush and commit | sending pending changes as SQL, and making them permanent | `session.commit()` | **The Session** |
| Identity map | one object for one row, however often it is loaded | `session.get(Student, 1)` | **The Identity Map** |
| Relationship | an attribute that reaches related rows, such as a student's enrollments | `student.enrollments` | **Relationships** |
| Association object | a class for a table that links two others and holds data of its own | an enrollment with a grade | **Many to Many** |
| Loading strategy | when related rows are read: on first use, or up front | `selectinload(...)` | **Loading Strategies** |
| Join and aggregate | a query across tables, and a total per group | `func.count()`, `group_by()` | **Joins and Aggregates** |
| Cascade | what happens to related objects when one is deleted | `cascade="all, delete-orphan"` | **Cascades and Deletes** |

Around both:

| Term | What it is | Example | Covered in depth in |
|---|---|---|---|
| `AsyncSession` | the ORM for code written with `async` and `await` | `await session.execute(...)` | **Async SQLAlchemy** |
| Migration | a numbered, reversible change to a schema already in use | `alembic upgrade head` | **Migrations with Alembic** |
| Test fixture | a database a test can change and have put back | a transaction rolled back after every test | **Testing a Data Layer** |
| Data layer | the models, migrations, loaders, queries and tests that own a program's database | `build.py`, which makes one from nothing | **A Complete Data Layer** |

### What this notebook covers

- The registrar's search in plain `sqlite3`, with placeholders, and what it still costs
- The same search with SQLAlchemy Core, built from `.where()` calls
- The SQL a statement becomes, for SQLite, PostgreSQL and SQL Server
- SQL written by hand, with `text()` and bound values
- Rows as objects: a first mapped class and a session
- When to use `sqlite3`, Core, the ORM or `text()`
- The search page, finished
- Seven errors, from an apostrophe in an f-string to a `.where()` whose result was thrown away

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import Column, Integer, MetaData, String, Table, create_engine, select

engine = create_engine("sqlite://")
metadata = MetaData()
students = Table("students", metadata, Column("id", Integer, primary_key=True),
                 Column("name", String), Column("program", String))
metadata.create_all(engine)

with engine.begin() as conn:
    conn.execute(students.insert(), [{"name": "Aoife O'Brien", "program": "History"},
                                     {"name": "Ben Okafor", "program": "Biology"}])
    query = select(students.c.name).where(students.c.name.contains("O'Brien"))
    print(" ".join(str(query).split()))          # the SQL, on one line
    print(conn.execute(query).all())
```

```
SELECT students.name FROM students WHERE (students.name LIKE '%' || :name_1 || '%')
[("Aoife O'Brien",)]
```

The table, the rows and the query are all Python objects. The query's SQL has a bind parameter,
`:name_1`, where the name would have been pasted, so `O'Brien` went to the database as a value, and
the apostrophe did no harm.


## Setup

Eight imports, the college's database, and a helper that prints the SQL a statement becomes.

- `sqlite3`, from the standard library, builds the database, and runs the search written as SQL text
- `sqlalchemy` is the library itself, and the cell prints its version
- `create_engine`, `MetaData`, `Table`, `Column`, the types `Integer`, `String` and `Date`, `select`
  and `text`, from `sqlalchemy`, are Core
- `postgresql` and `mssql`, from `sqlalchemy.dialects`, write SQL for PostgreSQL and SQL Server,
  with no server and no connection
- `DeclarativeBase`, `Mapped`, `mapped_column` and `Session`, from `sqlalchemy.orm`, are the ORM
- `date` is what a column declared as a `Date` gives back
- `Path` names the scratch folder and the database file in it
- `shutil` removes the scratch folder at the start and at the end

The database, `scratch/college.db`, holds 25 students and 10 courses. `show_sql` prints what a
statement becomes for one database, a line at a time, with the values that travel beside it.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import shutil
import sqlite3
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import Column, Date, Integer, MetaData, String, Table, create_engine, select, text
from sqlalchemy.dialects import mssql, postgresql
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
STARTS = ["2024-08-26", "2025-01-13", "2025-08-25"]           # the first day of three terms
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], STARTS[i % len(STARTS)]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.commit()
build.close()


def show_sql(statement, dialect):
    """Print the SQL a statement becomes for one database, and the values that travel beside it."""
    compiled = statement.compile(dialect=dialect)
    for line in str(compiled).splitlines():
        print("   ", line.rstrip())
    print("    values:", compiled.params)


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "holds", len(STUDENTS), "students and", len(COURSES), "courses")


sqlalchemy 2.0.54 | scratch/college.db holds 25 students and 10 courses


## Worked examples

### The search page, in plain sqlite3

The registrar's search, written carefully: every value goes in through a `?` placeholder, never
pasted into the text. The three `if` statements each add a condition and its value, and the
conditions are joined with `AND` at the end:


In [2]:
def search_students_sql(conn, program=None, started_after=None, name=None):
    """The registrar's search, as SQL text with a placeholder for every value."""
    sql = "SELECT name, program, started_on FROM students"
    conditions, values = [], []
    if program:
        conditions.append("program = ?")
        values.append(program)
    if started_after:
        conditions.append("started_on >= ?")
        values.append(started_after)
    if name:
        conditions.append("name LIKE ?")
        values.append(f"%{name}%")
    if conditions:
        sql += " WHERE " + " AND ".join(conditions)
    return conn.execute(sql + " ORDER BY name", values).fetchall()


raw = sqlite3.connect(DATABASE)
print(search_students_sql(raw, name="O'Brien"))
print(search_students_sql(raw, program="Biology", started_after="2025-01-01"))
raw.close()


[("Aoife O'Brien", 'History', '2024-08-26')]
[('Felix Wagner', 'Biology', '2025-08-25'), ('Keiko Tanaka', 'Biology', '2025-01-13'), ('Umar Farouk', 'Biology', '2025-08-25')]


Both searches work, `O'Brien` included, since the name reached SQLite as a value. What is left is
bookkeeping: two lists that must stay in step, a `WHERE` added only when there is something to add,
and SQL that exists only as text assembled at run time. The rows are tuples, so the program is
`row[1]` for as long as nobody reorders the columns, and `'2025-08-25'` is text, not a date.

### The same search, with SQLAlchemy Core

An engine for the same file, a `Table` that describes the `students` table, and the search built as a
statement. Every `.where()` returns a new statement with one more condition, so the result of each
call is kept:


In [3]:
engine = create_engine(f"sqlite:///{DATABASE}")
metadata = MetaData()
students = Table(
    "students", metadata,
    Column("id", Integer, primary_key=True),
    Column("name", String),
    Column("email", String),
    Column("program", String),
    Column("started_on", Date),
)


def search_students_core(conn, program=None, started_after=None, name=None):
    """The registrar's search, built as a Core statement."""
    query = select(students.c.name, students.c.program, students.c.started_on)
    if program:
        query = query.where(students.c.program == program)
    if started_after:
        query = query.where(students.c.started_on >= started_after)
    if name:
        query = query.where(students.c.name.contains(name))
    return conn.execute(query.order_by(students.c.name)).all()


with engine.connect() as conn:
    for row in search_students_core(conn, program="Biology", started_after=date(2025, 1, 1)):
        print(row.name, "|", row.program, "|", repr(row.started_on))


Felix Wagner | Biology | datetime.date(2025, 8, 25)
Keiko Tanaka | Biology | datetime.date(2025, 1, 13)
Umar Farouk | Biology | datetime.date(2025, 8, 25)


The same three students, and three differences. There is no SQL text to keep in step with anything:
`students.c.program == program` is a condition, not a string, and `students.c` holds the table's
columns. The rows answer to column names, `row.name` rather than `row[0]`. And `started_on` came back
as a `date`, because the `Table` declared that column a `Date`, which also let the search take a
`date` as its first day. The engine's URL, `sqlite:///` followed by the path, says which database
this is, and the **Engines and URLs** notebook takes URLs apart.

### The SQL a statement becomes

A statement is an object until it runs, so it can be printed first. `compile` turns it into the SQL
of one database, and the same statement gives different SQL for different databases:


In [4]:
query = (
    select(students.c.name, students.c.program)
    .where(students.c.program == "Biology")
    .order_by(students.c.name)
    .limit(5)
)

print("for SQLite, the engine's own dialect:")
show_sql(query, engine.dialect)
print("for PostgreSQL:")
show_sql(query, postgresql.dialect())
print("for SQL Server, with the values written in, for reading only:")
print("   ", " ".join(str(query.compile(dialect=mssql.dialect(), compile_kwargs={"literal_binds": True})).split()))


for SQLite, the engine's own dialect:
    SELECT students.name, students.program
    FROM students
    WHERE students.program = ? ORDER BY students.name
     LIMIT ? OFFSET ?
    values: {'program_1': 'Biology', 'param_1': 5, 'param_2': 0}
for PostgreSQL:
    SELECT students.name, students.program
    FROM students
    WHERE students.program = %(program_1)s ORDER BY students.name
     LIMIT %(param_1)s
    values: {'program_1': 'Biology', 'param_1': 5}
for SQL Server, with the values written in, for reading only:
    SELECT TOP 5 students.name, students.program FROM students WHERE students.program = 'Biology' ORDER BY students.name


SQLite's driver takes `?` placeholders, and PostgreSQL's takes `%(program_1)s`, and the values
travel beside the SQL either way, named in `values`. For SQLite, SQLAlchemy writes an `OFFSET` of 0
beside the `LIMIT`, which changes nothing. SQL Server has no `LIMIT` at all, and writes `TOP 5` at
the front instead. None of it needed a server: a dialect only writes SQL. `literal_binds` writes the
values into the text, which is for a person reading it and never for running, since it is exactly
the pasting that bind parameters exist to avoid.

### SQL you write yourself: text()

Some SQL is easier written by hand. `text()` marks a string as SQL, and a name after a colon is a
bind parameter, whose value goes in a dictionary beside it:


In [5]:
PER_PROGRAM = text("SELECT COUNT(*) FROM students WHERE program = :program")

with engine.connect() as conn:
    for program in ["History", "Mathematics"]:
        print(program, conn.execute(PER_PROGRAM, {"program": program}).scalar_one())


History 5
Mathematics 5


The same `text()` statement ran twice with two values, and neither was ever pasted into it.
`scalar_one()` takes the single value of a single row, and the **Reading Results** notebook covers
it with the other ways of reading a result. `text()` is Core too: it has bind parameters, and a
dialect, but none of the building, since SQLAlchemy does not look inside the string.

### Rows as objects: a first mapped class

The ORM maps a class to a table. `DeclarativeBase` is the base for a family of mapped classes,
`__tablename__` names the table, and every annotated attribute is a column, typed by its annotation.
A `Session` runs ORM statements, and `scalars` hands back the first thing each row holds, which for
`select(Student)` is a `Student`:


In [6]:
class Base(DeclarativeBase):
    pass


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    email: Mapped[str]
    program: Mapped[str]
    started_on: Mapped[date]

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


with Session(engine) as session:
    for student in session.scalars(select(Student).where(Student.program == "History").order_by(Student.name)):
        print(student, "started", student.started_on, "|", student.email)
    print("started_on is a", type(student.started_on).__name__)


Student("Aoife O'Brien", 'History') started 2024-08-26 | aobrien@college.edu
Student('Elena Petrova', 'History') started 2025-01-13 | epetrova@college.edu
Student('Jonas Berg', 'History') started 2024-08-26 | jberg@college.edu
Student('Olivia Brandt', 'History') started 2025-08-25 | obrandt@college.edu
Student('Tara Nilsen', 'History') started 2025-01-13 | tnilsen@college.edu
started_on is a date


Five `Student` objects, each with its columns as attributes, and `started_on` a `date` again, from
the `Mapped[date]` annotation alone. `Student.program == "History"` is the same kind of condition as
before, written on the class instead of on a `Table`, since mapping the class built a `Table` for
it. The `__repr__` is there because a mapped class prints as its memory address without one. The
**Declarative Models** notebook covers mapping, and **The Session** what a session does with the
objects it hands out.

### sqlite3, Core, the ORM, or text()

| Write | When | Why |
|---|---|---|
| `sqlite3` alone | a script, one SQLite file, and queries that never change shape | nothing to install, and every line of SQL in plain sight |
| Core: `Table` and `select()` | queries built from options, reports, loads, and anything that may run on another database | the statement is an object, values are always bound, and the dialect writes the SQL |
| the ORM: mapped classes and a `Session` | an application whose rows are things with behavior, such as students and their enrollments | rows come back as objects, and the session writes their changes back |
| `text()` | a statement easier to write than to build, inside code that already uses SQLAlchemy | SQL by hand, with bind parameters still doing the work |

The default for an application is the ORM, with Core underneath for reports and bulk work, and
`text()` for the odd statement that reads better as SQL. `sqlite3` alone suits a script that will
never be anything more.

### The search page, finished

The pieces of this notebook in one function: the registrar's three boxes, and a fourth, how many
results a page shows. It builds an ORM statement with a `.where()` for every box that was filled in,
orders the result and limits it, and returns `Student` objects:


In [7]:
def search_students(session, program=None, started_after=None, name=None, limit=10):
    """The registrar's search: the students matching every box that was filled in, by name."""
    query = select(Student)
    if program:
        query = query.where(Student.program == program)
    if started_after:
        query = query.where(Student.started_on >= started_after)
    if name:
        query = query.where(Student.name.contains(name))
    return session.scalars(query.order_by(Student.name).limit(limit)).all()


with Session(engine) as session:
    print(search_students(session, name="O'Brien"))
    print(search_students(session, program="Biology", started_after=date(2025, 1, 1)))
    print(search_students(session, limit=3))

show_sql(select(Student).where(Student.program == "Biology", Student.started_on >= date(2025, 1, 1)), engine.dialect)


[Student("Aoife O'Brien", 'History')]
[Student('Felix Wagner', 'Biology'), Student('Keiko Tanaka', 'Biology'), Student('Umar Farouk', 'Biology')]
[Student('Ana Reyes', 'Biology'), Student("Aoife O'Brien", 'History'), Student('Ben Okafor', 'Computer Science')]
    SELECT students.id, students.name, students.email, students.program, students.started_on
    FROM students
    WHERE students.program = ? AND students.started_on >= ?
    values: {'program_1': 'Biology', 'started_on_1': datetime.date(2025, 1, 1)}


Three searches through one function, the first with an apostrophe in it, the second with two boxes
filled in, and the third with none, which returns the first three students by name. The last lines
print the statement the second search ran: two conditions, joined by `AND`, and both values bound.
`.where()` takes several conditions at once, too, and joins them the same way.

### Where each part came from

| In the search | What it relies on | The section that showed it |
|---|---|---|
| `query = query.where(...)` for every box | a statement built up one condition at a time, the result kept each time | The same search, with SQLAlchemy Core |
| `Student.program == program` | a condition with its value bound, never pasted into SQL | The same search, with SQLAlchemy Core |
| `select(Student)` and `session.scalars` | rows returned as objects of a mapped class | Rows as objects: a first mapped class |
| `Student.started_on >= started_after` | a `Date` column that takes and returns `date` objects | The same search, with SQLAlchemy Core |
| `show_sql` | a statement printed as the SQL one database receives | The SQL a statement becomes |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/01-why-sqlalchemy-solutions.ipynb).

**1.** With `search_students`, list the History students who started on or after 1 January 2025.


In [8]:
# your code here


**2.** Print, with `show_sql`, the SQL of a search with all three boxes filled in, for SQLite.


In [9]:
# your code here


**3.** Print the same statement for PostgreSQL, and say what its placeholders look like.


In [10]:
# your code here


**4.** With `text()`, count the students who started on each of the three first days, in one
statement with `GROUP BY`.


In [11]:
# your code here


**5.** Give `search_students` a `page` argument, so that page 2 of a search with a limit of 5 returns
the sixth to the tenth students, and print the first two pages.


In [12]:
# your code here


**6.** Map the `courses` table as a `Course` class, and print the courses worth 4 credits, by code.


In [13]:
# your code here


## Common errors

### sqlite3.OperationalError: near "Brien": syntax error


In [14]:
def search_by_name(conn, name):
    """A search with the name pasted into the SQL, which is the mistake."""
    return conn.execute(f"SELECT name FROM students WHERE name LIKE '%{name}%'").fetchall()


raw = sqlite3.connect(DATABASE)
print(search_by_name(raw, "Okafor"))
search_by_name(raw, "O'Brien")


[('Ben Okafor',)]


OperationalError: near "Brien": syntax error

The first search worked. The second pasted `O'Brien` into the text, and its apostrophe closed the
quoted pattern early, so SQLite read `Brien` as SQL and refused it. The same hole lets a search box
change what the statement does. Keep values out of the text, with a placeholder, or with a
SQLAlchemy condition, which binds the value for you:


In [15]:
raw.close()

with engine.connect() as conn:
    print(conn.execute(select(students.c.name).where(students.c.name.contains("O'Brien"))).all())


[("Aoife O'Brien",)]


### sqlalchemy.exc.ObjectNotExecutableError: Not an executable object: 'SELECT 1'


In [16]:
with engine.connect() as conn:
    conn.execute("SELECT 1")


ObjectNotExecutableError: Not an executable object: 'SELECT 1'

A SQLAlchemy connection runs statements, and a plain string is not one: SQLAlchemy 2.0 will not guess
that text is meant as SQL, since text built from input is exactly where injection comes from. Say so
with `text()`:


In [17]:
with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).scalar_one())


1


### sqlalchemy.exc.StatementError: (sqlalchemy.exc.InvalidRequestError) A value is required for bind parameter 'n'


In [18]:
FIRST_N = text("SELECT name FROM students ORDER BY name LIMIT :n")

with engine.connect() as conn:
    conn.execute(FIRST_N).all()


StatementError: (sqlalchemy.exc.InvalidRequestError) A value is required for bind parameter 'n'
[SQL: SELECT name FROM students ORDER BY name LIMIT ?]
(Background on this error at: https://sqlalche.me/e/20/cd3x)

`:n` in a `text()` statement is a bind parameter, and a statement cannot run with one of its values
missing. The error names the parameter, and shows the SQL it was sent to underneath, with `?` where
`:n` was, since SQLite's driver takes `?`. Pass the values in a dictionary, a key for every name:


In [19]:
with engine.connect() as conn:
    print(conn.execute(FIRST_N, {"n": 3}).all())


[('Ana Reyes',), ("Aoife O'Brien",), ('Ben Okafor',)]


### sqlalchemy.exc.ArgumentError: Could not parse SQLAlchemy URL from given URL string


In [20]:
create_engine("scratch/college.db")


ArgumentError: Could not parse SQLAlchemy URL from given URL string

`create_engine` takes a URL, not a path, and a path has none of a URL's parts: the kind of database
before `://`, and where to find it after. For a SQLite file, the URL is `sqlite:///` followed by the
path, three slashes for a path relative to the current folder:


In [21]:
print(create_engine(f"sqlite:///{DATABASE}").url)


sqlite:///scratch/college.db


### AttributeError: 'Engine' object has no attribute 'execute'


In [22]:
engine.execute(text("SELECT COUNT(*) FROM students"))


AttributeError: 'Engine' object has no attribute 'execute'

Older tutorials run SQL straight from the engine. SQLAlchemy 1.4 still allowed it, marked for
removal, and 2.0 removed it: a statement runs on a connection, which says when it starts and ends,
and the **Connections and Transactions** notebook shows why that matters. Open a connection and run
the statement on it:


In [23]:
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM students")).scalar_one())


25


### sqlalchemy.exc.ArgumentError: Textual SQL expression "program = 'Biology'" should be explicitly declared as text("program = 'Biology'")


In [24]:
select(students.c.name).where("program = 'Biology'")


ArgumentError: Textual SQL expression "program = 'Biology'" should be explicitly declared as text("program = 'Biology'")

`.where()` takes conditions built from columns, and a string is not one. SQLAlchemy refuses to treat
it as SQL, for the same reason `execute` refuses a plain string, and even says how it could be
written. Better is a condition on the column, which binds the value instead of trusting the text:


In [25]:
with engine.connect() as conn:
    print(conn.execute(select(students.c.name).where(students.c.program == "Biology").limit(2)).all())


[('Ana Reyes',), ('Felix Wagner',)]


### No error, and every student in the results: a .where() whose statement was thrown away


In [26]:
def search_by_program(session, program=None):
    """A search that calls .where() and keeps the statement it started with, which is the mistake."""
    query = select(Student).order_by(Student.name)
    if program:
        query.where(Student.program == program)
    return session.scalars(query).all()


with Session(engine) as session:
    print(len(search_by_program(session, program="History")), "students for History")


25 students for History


There are five History students, and the search returned all 25. A statement never changes once it
is built: `.where()` returned a new statement with the condition in it, and nothing kept that new
statement, so the query that ran was the one with no condition at all. Nothing raised, since the
code is correct Python and the SQL is correct SQL. Assign the result, every time:


In [27]:
def search_by_program(session, program=None):
    """The same search, keeping the statement that .where() returns."""
    query = select(Student).order_by(Student.name)
    if program:
        query = query.where(Student.program == program)
    return session.scalars(query).all()


with Session(engine) as session:
    print(len(search_by_program(session, program="History")), "students for History")


5 students for History


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [28]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- SQLAlchemy writes SQL as Python objects in Core, and maps tables to classes in the ORM, which runs
  Core underneath.
- A statement is an object: `.where()` returns a new statement with one more condition, joined by
  `AND`, so a query built from options keeps the result of every call.
- Values travel as bind parameters, never inside the SQL, so a name with an apostrophe is just a
  name.
- An engine made from a URL reaches one database, and its dialect writes the SQL that database
  speaks, which `compile` shows for any dialect with no server.
- `text()` marks a string as SQL and keeps its values bound, and nothing else is run as SQL.
- A mapped class turns rows into objects, with column types converting values both ways.


## What is next

The **Engines and URLs** notebook takes the engine apart: what a URL names, the dialect and the
driver behind it, `echo` for watching the SQL an engine sends, the pool of connections it keeps
without being asked, and the database in memory that looks empty from another thread.


---

[SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Engines and URLs](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/02-engines-and-urls.ipynb) &#8594;
